In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [ ]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
opt = torch_numopt.LM(
    params=model.parameters(),
    model=model,
    lr=1,
    mu=0.001,
    mu_dec=0.1,
    fletcher=False,
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo",
    solver="solve",
    batch_size=100,
    # batch_size=None,
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.6733808517456055
epoch:  1, loss: 0.6518338918685913
epoch:  2, loss: 0.6445268392562866
epoch:  3, loss: 0.6398177742958069
epoch:  4, loss: 0.6361804604530334
epoch:  5, loss: 0.6331655979156494
epoch:  6, loss: 0.6309229731559753
epoch:  7, loss: 0.6290106177330017
epoch:  8, loss: 0.6274344325065613
epoch:  9, loss: 0.6260560154914856
epoch:  10, loss: 0.6248708367347717
epoch:  11, loss: 0.6238174438476562
epoch:  12, loss: 0.6228721737861633
epoch:  13, loss: 0.6220033168792725
epoch:  14, loss: 0.6211796998977661
epoch:  15, loss: 0.6204268932342529
epoch:  16, loss: 0.6197172403335571
epoch:  17, loss: 0.619047999382019
epoch:  18, loss: 0.6184054613113403
epoch:  19, loss: 0.6177835464477539
epoch:  20, loss: 0.6172024607658386
epoch:  21, loss: 0.6166520714759827
epoch:  22, loss: 0.6161296963691711
epoch:  23, loss: 0.615626871585846
epoch:  24, loss: 0.6151400208473206
epoch:  25, loss: 0.6146813035011292
epoch:  26, loss: 0.6142332553863525
epoch:  27, l

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -22595.272899873384
Test metrics:  R2 = -15951.42719732025


In [ ]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
opt = torch_numopt.LM(
    params=model.parameters(),
    model=model,
    lr=1,
    mu=0.001,
    mu_dec=0.1,
    fletcher=True,
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo",
    solver="pinv",
    batch_size=100,
    # batch_size=None,
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.68842613697052
epoch:  1, loss: 0.6884210109710693
epoch:  2, loss: 0.6884159445762634
epoch:  3, loss: 0.688410758972168
epoch:  4, loss: 0.6884057521820068
epoch:  5, loss: 0.6884006261825562
epoch:  6, loss: 0.6883955001831055
epoch:  7, loss: 0.6883904337882996
epoch:  8, loss: 0.6883853077888489
epoch:  9, loss: 0.688380241394043
epoch:  10, loss: 0.6883751749992371
epoch:  11, loss: 0.6883700489997864
epoch:  12, loss: 0.6883649826049805
epoch:  13, loss: 0.6883599758148193
epoch:  14, loss: 0.6883549094200134
epoch:  15, loss: 0.6883498430252075
epoch:  16

KeyboardInterrupt: 

In [ ]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7086681734630786
Test metrics:  R2 = 0.6902413934940552
